# 3.15 Random Values

Completed Popcorn Hacks and homework for the [Python lesson](https://pages.opencodingsociety.com/python/random/2027). A fixed seed makes this notebook's example reproducible; the simulator still uses random selections.


## Popcorn Hack 1: Random SFI record number

Select an integer from 1 through 4. Repeated record numbers are possible because a random draw does not remove a value from the choices.


In [1]:
import random

rng = random.Random(17)
record_count = 4
record_numbers = [rng.randint(1, record_count) for _ in range(20)]
print(record_numbers)
assert all(1 <= number <= record_count for number in record_numbers)
assert len(set(record_numbers)) < len(record_numbers)


[4, 3, 3, 3, 2, 3, 1, 1, 2, 4, 4, 3, 3, 4, 2, 1, 2, 2, 2, 2]


## Popcorn Hack 2: Random structured SFI car part

Select one complete record and display fields from that same object so that product, category, and spec stay together.


In [2]:
parts = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]
selected_part = random.Random(23).choice(parts)
print(selected_part["product_name"])
print(selected_part["category"])
print(selected_part["spec_number"])
assert selected_part in parts


Multiple Disc Clutch Assemblies
Drag Racing
1.2


## Popcorn Hack 3: Random SFI API test

Choose a part and a supported action randomly, then form the matching backend-style response.


In [3]:
def api_test_response(action, part):
    if action == "GET/search":
        return f"200 OK: found {part['product_name']} ({part['spec_number']})"
    if action == "POST/create":
        return f"201 Created: {part['spec_number']} queued for creation"
    if action == "PUT/update":
        return f"200 OK: {part['spec_number']} queued for update"
    raise ValueError(f"Unsupported action: {action}")

api_rng = random.Random(31)
selected_part = api_rng.choice(parts)
selected_action = api_rng.choice(["GET/search", "POST/create", "PUT/update"])
response = api_test_response(selected_action, selected_part)
print(selected_action, response)
assert response.startswith(("200 OK:", "201 Created:"))


POST/create 201 Created: 1.1 queued for creation


## Homework Hack: SFI backend QA simulator

Run five reproducible tests across search, create, update, and delete. The create handler rejects a spec number already in the catalog. The action order and selected record are randomized; the fixed seed makes this example repeatable.


In [4]:
def run_backend_action(action, part, catalog):
    spec = part["spec_number"]
    match_index = next((i for i, item in enumerate(catalog) if item["spec_number"] == spec), None)
    if action == "GET/search":
        return "200 OK: found" if match_index is not None else "404 Not Found"
    if action == "POST/create":
        if match_index is not None:
            return "409 Conflict: duplicate spec number"
        catalog.append(part.copy())
        return "201 Created"
    if action == "PUT/update":
        if match_index is None:
            return "404 Not Found"
        catalog[match_index]["product_name"] = part["product_name"]
        return "200 OK: updated"
    if action == "DELETE/remove":
        if match_index is None:
            return "404 Not Found"
        catalog.pop(match_index)
        return "200 OK: removed"
    raise ValueError(f"Unsupported action: {action}")

catalog = [parts[0].copy(), parts[2].copy()]
existing_spec_numbers = [item["spec_number"] for item in catalog]
assert run_backend_action("POST/create", parts[0], catalog) == "409 Conflict: duplicate spec number"
assert existing_spec_numbers == ["1.1", "2.1"]

simulation_rng = random.Random(41)
actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]
simulation_rng.shuffle(actions)
actions.append(simulation_rng.choice(actions))
test_count = 5
results = []
for index in range(test_count):
    action = actions[index]
    part = simulation_rng.choice(parts)
    result = run_backend_action(action, part, catalog)
    results.append((part["spec_number"], action, result))
    print(f"Run {index + 1}: {part['spec_number']} {action} -> {result}")
assert len(results) == test_count
assert {result[1] for result in results[:4]} == {"GET/search", "POST/create", "PUT/update", "DELETE/remove"}


Run 1: 1.2 PUT/update -> 404 Not Found
Run 2: 2.1 GET/search -> 200 OK: found
Run 3: 2.1 POST/create -> 409 Conflict: duplicate spec number
Run 4: 1.2 DELETE/remove -> 404 Not Found
Run 5: 2.1 GET/search -> 200 OK: found
